In [ ]:
import importlib.util, subprocess, sys
_missing = [p for p in ('huggingface_hub', 'tqdm', 'pandas', 'matplotlib') if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=True)
    print('installed', _missing, '- now Restart the kernel once, then Run All')
else:
    print('all packages present')

In [ ]:
import os, re, json, math, random, hashlib, zipfile, gc, time, pickle, warnings, sys, copy, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

SMOKE_TEST = False
SEED = 42
RESUME_COMPLETED_RUNS = True
WRITE_SUBMISSIONS = True
RUN_REPRESENTATION_STUDY = True

DATASET_ID = "Exploration-Lab/CS-781-Capstone"
DATA_SOURCE = "hf"
LOCAL_DATA_DIR = None
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "."]
PHASE1_CKPT_OVERRIDE = None
TOKENIZER_PATH_OVERRIDE = None
TOP_K = 7

BM25_K1_GRID = (0.9, 1.2, 1.6, 2.0)
BM25_B_GRID = (0.5, 0.75, 0.9)
BM25_NGRAMS_GRID = ((1,), (1, 2))
HARD_NEG_POOL = 50

CHUNK_TOKENS = 512
N_CHUNKS = 4
EVAL_N_CHUNKS = 8
SELECTION = "spread"
HEAD_CHUNKS = 1
MAX_WORDS = 20000
XL_MEMORY = 0
XL_MEMORY_OLD = 0
CHUNK_POOLING = "mean+last"
EMB_DIM = 256
N_POS_BUCKETS = 8

EPOCHS = 2
MAX_TRAIN_MINUTES = 45
PAIRS_PER_BATCH = 4
HARD_NEGS = 2
GRAD_ACCUM = 2
EVAL_BATCH_DOCS = 8
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
HEAD_DROPOUT = 0.1
BACKBONE_DROPOUT = 0.1
INIT_TEMPERATURE = 0.05
USE_GRAD_CHECKPOINTING = True

EXPERIMENT_LIBRARY = {
    "lorma_plus": dict(method="lorma_plus", rank=16, alpha=16, lr=5e-4, head_lr=1e-3, patience=2,
                       description="LoRMA+ (compulsory): multiplicative low-rank update W = (I + a/r * B A) W0."),
    "ia3": dict(method="ia3", lr=3e-3, head_lr=1e-3, patience=2,
                description="IA3: learned rescaling of attention keys/values and FFN activations (~0.1% parameters)."),
    "qlora": dict(method="qlora", rank=16, alpha=32, lr=5e-4, head_lr=1e-3, patience=2,
                  description="QLoRA: frozen Phase-1 weights stored in 4-bit NF4 (block 64, absmax) + LoRA on top."),
    "partial": dict(method="partial", top_layers=4, llrd=0.85, lr=5e-5, head_lr=1e-3, patience=2,
                    description="Partial / layer-wise fine-tuning: top 4 layers trained, layer-wise LR decay 0.85."),
    "prefix": dict(method="prefix", prefix_len=32, prefix_hidden=256, lr=1e-3, head_lr=1e-3, patience=2,
                   description="Prefix tuning: trainable per-layer prefix states (MLP re-parameterised) in every layer's memory."),
    "full": dict(method="full", lr=2e-5, head_lr=1e-3, patience=2, description="Full fine-tuning (reference)."),
    "frozen": dict(method="frozen", lr=0.0, head_lr=1e-3, patience=2,
                   description="Frozen Phase-1 backbone, only the projection head trained (Phase-1 transfer probe)."),
    "distill": dict(method="distill", kd_temperature=1.0, kd_alpha=0.5, kd_emb_weight=1.0, lr=5e-5, head_lr=1e-3,
                    patience=2, description="Knowledge distillation: fine-tuned teacher retriever -> smaller student."),
}
EXPERIMENT_DEFAULTS = dict(hard_negs=None, selection=None, epochs=None, dropout=None, use_amp=True)

def make_experiment(base, **over):
    cfg = copy.deepcopy(EXPERIMENT_LIBRARY[base])
    cfg.update(over)
    return cfg

ARCH_NAME = "encoder_decoder"
OUTPUT_NAME = "phase7_encoder_decoder_pcr"
MIN_PARAMS, MAX_PARAMS = 100e6, 300e6
PHASE1_ARCH_KEYWORD = "encoder_decoder_transformer_rope_gqa_rmsnorm"
PAIRS_PER_BATCH, GRAD_ACCUM = 2, 4
LOWRANK_TARGETS = ("self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.out_proj",
                   "cross_attn.q_proj", "cross_attn.k_proj", "cross_attn.v_proj", "cross_attn.out_proj",
                   "ff.gate_proj", "ff.up_proj", "ff.down_proj", "ff.net.0", "ff.net.2")
ADAPTER_TARGETS = ("self_attn.out_proj", "cross_attn.out_proj", "ff.down_proj", "ff.net.2")
IA3_TARGETS = {"self_attn.k_proj": "out", "self_attn.v_proj": "out", "cross_attn.k_proj": "out",
               "cross_attn.v_proj": "out", "ff.down_proj": "in", "ff.net.2": "in"}
EXPERIMENTS = {
    "E2_ia3": make_experiment("ia3"),
    "E3_partial_ft": make_experiment("partial"),
}
RUN_LIST = ["E2_ia3", "E3_partial_ft"]

OUTPUT_DIR = Path("/kaggle/working") / OUTPUT_NAME if Path("/kaggle/working").exists() else Path(OUTPUT_NAME)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "runs").mkdir(exist_ok=True)

if SMOKE_TEST:
    CHUNK_TOKENS, N_CHUNKS, EVAL_N_CHUNKS, EMB_DIM = 32, 2, 3, 32
    PAIRS_PER_BATCH, HARD_NEGS, GRAD_ACCUM, EVAL_BATCH_DOCS, EPOCHS = 2, 1, 1, 4, 2
    USE_GRAD_CHECKPOINTING = False
    XL_MEMORY = min(XL_MEMORY, 16)
    XL_MEMORY_OLD = min(XL_MEMORY_OLD, 16)
    MIN_PARAMS = 0
    BM25_K1_GRID, BM25_B_GRID = (1.2,), (0.75,)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          "| memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
elif not SMOKE_TEST:
    raise RuntimeError("Enable a GPU accelerator (Kaggle: Settings > Accelerator > GPU T4/P100).")
print("Experiments to run:", RUN_LIST)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def save_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

def make_grad_scaler(enabled):
    if not enabled:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)

In [ ]:
class ByteBPE:
    EOW = 260

    def __init__(self):
        self.specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos, self.stoi, self.payload, self.merges = [], {}, {}, []
        self.vocab_size = 0
        self._ranks = {}
        self._cache = {}

    @classmethod
    def from_json(cls, payload):
        t = cls()
        t.itos = t.specials + [f"<0x{i:02X}>" for i in range(256)] + ["</w>"]
        t.stoi = {s: i for i, s in enumerate(t.itos)}
        t.payload = {i: bytes([i - 4]) for i in range(4, 260)}
        t.payload[260] = b""
        t.merges = [tuple(map(int, p)) for p in payload["merges"]]
        for pair in t.merges:
            new_id = len(t.itos)
            symbol = f"<{pair[0]}+{pair[1]}>"
            t.itos.append(symbol)
            t.stoi[symbol] = new_id
            t.payload[new_id] = t.payload.get(pair[0], b"") + t.payload.get(pair[1], b"")
        t.vocab_size = len(t.itos)
        if t.itos != payload.get("itos"):
            raise ValueError("Saved ByteBPE vocabulary and merge list are inconsistent.")
        t._ranks = {pair: r for r, pair in enumerate(t.merges)}
        return t

    def encode_word_reference(self, word):
        seq = [b + 4 for b in str(word).encode("utf-8")] + [self.EOW]
        for rank, pair in enumerate(self.merges):
            out, j = [], 0
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == pair:
                    out.append(261 + rank); j += 2
                else:
                    out.append(seq[j]); j += 1
            seq = out
        return seq

    def encode_piece(self, piece, add_eow=True):
        key = (piece, add_eow)
        hit = self._cache.get(key)
        if hit is not None:
            return hit
        seq = [b + 4 for b in piece.encode("utf-8")]
        if add_eow:
            seq.append(self.EOW)
        ranks = self._ranks
        while len(seq) > 1:
            best_rank, best_pair = None, None
            for i in range(len(seq) - 1):
                r = ranks.get((seq[i], seq[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, (seq[i], seq[i + 1])
            if best_pair is None:
                break
            out, j, new_id = [], 0, 261 + best_rank
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == best_pair:
                    out.append(new_id); j += 2
                else:
                    out.append(seq[j]); j += 1
            seq = out
        if len(self._cache) < 400_000:
            self._cache[key] = seq
        return seq

def _sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def _search(patterns, roots):
    found, seen = [], set()
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            for p in sorted(root.rglob(pat)):
                rp = str(p.resolve())
                if p.is_file() and rp not in seen:
                    seen.add(rp); found.append(p)
    return found

if PHASE1_CKPT_OVERRIDE:
    ckpt_candidates = [Path(PHASE1_CKPT_OVERRIDE)]
else:
    ckpt_candidates = _search(["best_t4*checkpoint*.pt", "best*checkpoint*.pt"], SEARCH_ROOTS) + \
                      _search(["latest*checkpoint*.pt", "*.pt", "*.pth"], SEARCH_ROOTS)
_payload, PHASE1_CKPT_PATH = None, None
for cand in ckpt_candidates:
    try:
        pl = torch.load(cand, map_location="cpu", weights_only=False)
    except Exception as exc:
        print(f"  skip {cand}: {exc}"); continue
    arch = str(pl.get("architecture", "")) if isinstance(pl, dict) else ""
    cfg_ = pl.get("model_config", {}) if isinstance(pl, dict) else {}
    if (isinstance(pl, dict) and "model_state_dict" in pl and PHASE1_ARCH_KEYWORD in arch.lower()
            and "encoder_layers" in cfg_ and "decoder_layers" in cfg_):
        _payload, PHASE1_CKPT_PATH = pl, cand
        break
    print(f"  skip {cand}: architecture={arch!r} (not {PHASE1_ARCH_KEYWORD})")
    del pl
if _payload is None:
    raise FileNotFoundError(
        "No RoPE-GQA-RMSNorm encoder-decoder Phase-1 checkpoint found. Attach the Phase-1 output containing "
        "best_t4_reused_bpe32k_checkpoint.pt "
        "('Add Input') or set PHASE1_CKPT_OVERRIDE.")
print("Phase-1 checkpoint:", PHASE1_CKPT_PATH)

P1_STATE = _payload["model_state_dict"]
P1_CFG = dict(_payload["model_config"])
P1_VOCAB = int(_payload.get("vocab_size", P1_CFG["vocab_size"]))
P1_TOKENIZER_SHA = _payload.get("tokenizer_sha256")
P1_ARCH = _payload.get("architecture")
P1_CFG.setdefault("rope_base", 10_000.0)
P1_CFG.setdefault("rms_norm_eps", 1e-6)
P1_CFG.setdefault("feed_forward", "swiglu")
print("Phase-1 architecture:", P1_ARCH)
print("Phase-1 config:", {k: P1_CFG.get(k) for k in ("d_model", "query_heads", "kv_heads", "encoder_layers", "decoder_layers",
                                                     "ff_hidden", "feed_forward", "encoder_max_length", "vocab_size",
                                                     "source_copy_head")})
assert P1_CFG.get("normalization", "RMSNorm") == "RMSNorm", "this notebook re-declares the RMSNorm variant"
del _payload
gc.collect()

if TOKENIZER_PATH_OVERRIDE:
    tok_candidates = [Path(TOKENIZER_PATH_OVERRIDE)]
else:
    tok_candidates = _search(["legal_bpe_tokenizer*.json", "*bpe*.json", "*tokenizer*.json"], SEARCH_ROOTS)

TOKENIZER, TOKENIZER_PATH, TOKENIZER_SHA256 = None, None, None
tok_table = []
for cand in tok_candidates:
    try:
        raw = json.loads(cand.read_text(encoding="utf-8"))
        if not (isinstance(raw, dict) and "merges" in raw and "itos" in raw):
            continue
        sha = _sha256_file(cand)
        tok_table.append((cand, int(raw.get("vocab_size", len(raw["itos"]))), sha, raw))
    except Exception:
        continue

print("Tokenizer files found:")
for cand, vs, sha, _ in tok_table:
    print(f"  {cand}  vocab={vs}  sha256={sha[:12]}")

chosen = [t for t in tok_table if P1_TOKENIZER_SHA and t[2] == P1_TOKENIZER_SHA]
how = "sha256 match with the Phase-1 checkpoint"
if not chosen:
    chosen = [t for t in tok_table if t[1] == P1_VOCAB]
    how = f"vocab size match ({P1_VOCAB}); sha256 differs/unknown - verify this is your Phase-1 tokenizer file"
if not chosen:
    raise FileNotFoundError(
        f"No tokenizer with vocab size {P1_VOCAB} found. Attach your legal_bpe_tokenizer JSON for that size "
        "(or set TOKENIZER_PATH_OVERRIDE). This notebook never trains a tokenizer.")
TOKENIZER_PATH, _, TOKENIZER_SHA256, _raw = chosen[0]
TOKENIZER = ByteBPE.from_json(_raw)
del tok_table, _raw
print(f"Using tokenizer: {TOKENIZER_PATH} ({how})")

PAD_ID, UNK_ID, BOS_ID, EOS_ID = (TOKENIZER.stoi[s] for s in ("<pad>", "<unk>", "<s>", "</s>"))
EOW_ID = ByteBPE.EOW
VOCAB_SIZE = TOKENIZER.vocab_size
assert VOCAB_SIZE == P1_VOCAB, f"Tokenizer vocab {VOCAB_SIZE} != checkpoint vocab {P1_VOCAB}"
print("Vocab size:", VOCAB_SIZE, "| ids:", dict(pad=PAD_ID, unk=UNK_ID, bos=BOS_ID, eos=EOS_ID, eow=EOW_ID))

_rng = random.Random(0)
_words = ["The", "Supreme", "Court", "of", "India,", "Section", "302", "IPC.", "(1995)", "Crl.A.No.123/2005",
          "appellant-State", "Hon'ble", "₹5,000", "naïve"]
for _w in _words:
    assert TOKENIZER.encode_piece(_w, True) == TOKENIZER.encode_word_reference(_w), _w
print("Fast BPE encoder == Phase-1 reference encoder on self-check words.")

In [ ]:
SPLITS = ("train", "dev", "test")
PHASE_RE = r"phase[_ -]?7|pcr"

def _pick(hits):
    pref = [h for h in hits if re.search(PHASE_RE, str(h), re.I)]
    return (pref or hits)[0]

def locate_phase7_files():
    names = {f"{s}_q": f"{s}.json" for s in SPLITS} | {f"{s}_c": f"{s}_candidates.json" for s in SPLITS}
    paths = {}
    if DATA_SOURCE == "local":
        roots = [LOCAL_DATA_DIR] if LOCAL_DATA_DIR else SEARCH_ROOTS
        for k, name in names.items():
            hits = _search([name], roots)
            if not hits:
                raise FileNotFoundError(f"{name} not found under {roots}")
            paths[k] = _pick(hits)
        return paths
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face read token: ")
    from huggingface_hub import HfApi, hf_hub_download
    repo_files = HfApi().list_repo_files(DATASET_ID, repo_type="dataset", token=token)
    for k, name in names.items():
        hits = [f for f in repo_files if (f.endswith("/" + name) or f == name) and re.search(PHASE_RE, f, re.I)]
        if not hits:
            raise FileNotFoundError(f"Phase-7 {name} not found in {DATASET_ID}; some files: {repo_files[:30]}")
        paths[k] = Path(hf_hub_download(DATASET_ID, hits[0], repo_type="dataset", token=token))
    return paths

P7_PATHS = locate_phase7_files()
for k, v in P7_PATHS.items():
    print(f"{k:8s} -> {v}")

def _load(path):
    raw = path.read_text(encoding="utf-8")
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return [json.loads(line) for line in raw.splitlines() if line.strip()]

def _text(v):
    if isinstance(v, dict):
        v = v.get("text", v.get("sentences", v.get("document", "")))
    if isinstance(v, list):
        return " ".join("" if t is None else str(t) for t in v)
    return "" if v is None else str(v)

def _as_records(obj):
    if isinstance(obj, dict):
        for key in ("data", "queries", "candidates", "documents", "cases"):
            if key in obj and isinstance(obj[key], list):
                return obj[key]
        return [dict(v, id=v.get("id", k)) if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()]
    return obj

def _id(d, i):
    return str(d.get("id", d.get("case_id", d.get("query_id", d.get("candidate_id", i)))))

QUERIES, POOLS = {}, {}
for s in SPLITS:
    qs = []
    for i, d in enumerate(_as_records(_load(P7_PATHS[f"{s}_q"]))):
        q = {"id": _id(d, i), "text": _text(d)}
        rel = d.get("relevant_candidates", d.get("relevant", d.get("labels")))
        if s != "test":
            if rel is None:
                raise KeyError(f"{s} query {q['id']}: no relevant_candidates; keys = {list(d)}")
            q["gold"] = [str(x) for x in rel]
        qs.append(q)
    QUERIES[s] = qs
    POOLS[s] = {_id(d, i): _text(d) for i, d in enumerate(_as_records(_load(P7_PATHS[f"{s}_c"])))}

test_ids = [q["id"] for q in QUERIES["test"]]
for s in SPLITS:
    pool = POOLS[s]
    w = np.array([len(t.split()) for t in pool.values()])
    msg = f"[{s:5s}] {len(QUERIES[s]):5,d} queries | pool {len(pool):6,d} cases (words: median {int(np.median(w)):,}, p90 {int(np.percentile(w, 90)):,})"
    if s != "test":
        g = np.array([len(q["gold"]) for q in QUERIES[s]])
        miss = sum(c not in pool for q in QUERIES[s] for c in q["gold"])
        selfp = sum(q["id"] in pool for q in QUERIES[s])
        msg += f" | gold per query: mean {g.mean():.2f}, median {int(np.median(g))}, max {g.max()} | gold not in pool: {miss} | queries also in their pool: {selfp}"
    print(msg)
print(f"Micro-F1@{TOP_K}: precision over the {TOP_K} returned ids, recall over ALL gold ids (not truncated).")

In [ ]:
TOKEN_DTYPE = np.uint16 if VOCAB_SIZE <= 65535 else np.int32

def tokenise(text):
    words = text.split()
    n_words = len(words)
    if n_words > MAX_WORDS:
        h = MAX_WORDS // 2
        words = words[:h] + words[-(MAX_WORDS - h):]
    ids = []
    enc = TOKENIZER.encode_piece
    for w in words:
        ids.extend(enc(w, True))
    return np.asarray(ids or [UNK_ID], dtype=TOKEN_DTYPE), n_words

_tok_cache = OUTPUT_DIR / (f"p7_tokens_{TOKENIZER_SHA256[:8]}_{MAX_WORDS}_"
                           f"{sum(len(QUERIES[s]) + len(POOLS[s]) for s in SPLITS)}.pkl")
DOCS = {}
if _tok_cache.exists():
    DOCS = pickle.loads(_tok_cache.read_bytes())
    print("Loaded cached tokenisation.")
else:
    for s in SPLITS:
        for q in tqdm(QUERIES[s], desc=f"tokenising {s} queries"):
            st, nw = tokenise(q["text"])
            DOCS[(s, "q", q["id"])] = dict(id=q["id"], stream=st, n_tokens=len(st), n_words=nw)
        for cid, t in tqdm(POOLS[s].items(), desc=f"tokenising {s} pool"):
            st, nw = tokenise(t)
            DOCS[(s, "c", cid)] = dict(id=cid, stream=st, n_tokens=len(st), n_words=nw)
    _tok_cache.write_bytes(pickle.dumps(DOCS, protocol=4))
_nt = np.array([d["n_tokens"] for k, d in DOCS.items() if k[1] == "c"])
print(f"candidate cases: tokens median {int(np.median(_nt)):,} | p90 {int(np.percentile(_nt, 90)):,} | "
      f"{EVAL_N_CHUNKS} x {CHUNK_TOKENS} = {EVAL_N_CHUNKS * CHUNK_TOKENS:,} tokens cover the whole case for "
      f"{np.mean(_nt <= EVAL_N_CHUNKS * CHUNK_TOKENS):.0%} of them")

def select_starts(n, mode, K, head_k=HEAD_CHUNKS, C=None):
    C = C or CHUNK_TOKENS
    if n <= C:
        return [0]
    if math.ceil(n / C) <= K:
        return sorted(set(list(range(0, n - C, C)) + [n - C]))
    if mode == "head":
        st = [i * C for i in range(K)]
    elif mode == "tail":
        st = [n - (K - i) * C for i in range(K)]
    elif mode == "head+tail":
        h = min(head_k, K - 1)
        st = [i * C for i in range(h)] + [n - (K - h - j) * C for j in range(K - h)]
    elif mode == "middle":
        s0 = int(min(max(0, n / 2 - K * C / 2), n - K * C))
        st = [s0 + i * C for i in range(K)]
    else:
        st = [int(round(x)) for x in np.linspace(0, n - C, K)]
    return sorted(set(max(0, min(s, n - C)) for s in st))

def make_doc_item(doc, mode, K):
    n = doc["n_tokens"]
    chunks = []
    for a in select_starts(n, mode, K):
        b = min(n, a + CHUNK_TOKENS)
        mem = doc["stream"][max(0, a - XL_MEMORY - XL_MEMORY_OLD):a] if XL_MEMORY else np.zeros(0, TOKEN_DTYPE)
        bucket = min(N_POS_BUCKETS - 1, int((a + b) / 2 / max(n, 1) * N_POS_BUCKETS))
        chunks.append(dict(ids=doc["stream"][a:b].astype(np.int64), start=a, mem=mem.astype(np.int64), bucket=bucket))
    return dict(chunks=chunks, id=doc["id"])

def collate_docs(batch):
    chunks = [c for it in batch for c in it["chunks"]]
    W = len(chunks)
    T = max(len(c["ids"]) for c in chunks)
    M = (XL_MEMORY + XL_MEMORY_OLD) if any(len(c["mem"]) for c in chunks) else 0
    ids = torch.full((W, T), PAD_ID, dtype=torch.long); mask = torch.zeros((W, T), dtype=torch.bool)
    mem_ids = torch.full((W, M), PAD_ID, dtype=torch.long); mem_mask = torch.zeros((W, M), dtype=torch.bool)
    starts = torch.zeros(W, dtype=torch.float32)
    c_doc, c_slot, c_bucket = [], [], []
    wi = 0
    for di, it in enumerate(batch):
        for si, c in enumerate(it["chunks"]):
            n = len(c["ids"])
            ids[wi, :n] = torch.from_numpy(c["ids"]); mask[wi, :n] = True
            k = len(c["mem"])
            if k:
                mem_ids[wi, M - k:] = torch.from_numpy(c["mem"]); mem_mask[wi, M - k:] = True
            starts[wi] = c["start"]
            c_doc.append(di); c_slot.append(si); c_bucket.append(c["bucket"])
            wi += 1
    t = lambda x: torch.tensor(x, dtype=torch.long)
    return dict(ids=ids, mask=mask, mem_ids=mem_ids, mem_mask=mem_mask, starts=starts, c_doc=t(c_doc),
                c_slot=t(c_slot), c_bucket=t(c_bucket), n_chunks=t([len(it["chunks"]) for it in batch]))

def to_device(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

In [ ]:
STOP = set("""a an the of to in on at by for with from and or but not no nor is are was were be been being this that these
those it its as which who whom whose what when where why how all any both each few more most other some such only own same
so than too very can will just should now he she they them his her their there here also into upon under over said such
shall may might must would could has have had do does did i we you our your us me my mr ms sri smt ld learned""".split())
_TOK = re.compile(r"[a-z0-9]+(?:\.[0-9]+)*")

def bm25_terms(text, ngrams):
    w = [t for t in _TOK.findall(text.lower()) if len(t) > 1 and t not in STOP]
    c = Counter(w)
    if 2 in ngrams:
        c.update(f"{a}_{b}" for a, b in zip(w, w[1:]))
    return c

class BM25Index:
    def __init__(self, split, ngrams):
        self.split, self.ngrams = split, ngrams
        self.cids = list(POOLS[split])
        counts = [bm25_terms(POOLS[split][c], ngrams) for c in tqdm(self.cids, desc=f"BM25 {split} {ngrams}", leave=False)]
        vocab = {}
        rows, cols, vals = [], [], []
        for i, c in enumerate(counts):
            for t, n in c.items():
                j = vocab.setdefault(t, len(vocab))
                rows.append(i); cols.append(j); vals.append(n)
        self.vocab = vocab
        self.tf = sp.csr_matrix((np.array(vals, np.float32), (rows, cols)), shape=(len(counts), len(vocab)))
        self.dl = np.asarray(self.tf.sum(1)).ravel()
        df = np.bincount(self.tf.indices, minlength=len(vocab))
        N = len(counts)
        self.idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        self.q = {}

    def query_matrix(self, qs):
        rows, cols, vals = [], [], []
        for i, q in enumerate(qs):
            for t, n in bm25_terms(q["text"], self.ngrams).items():
                j = self.vocab.get(t)
                if j is not None:
                    rows.append(i); cols.append(j); vals.append(1.0 + math.log(n))
        return sp.csr_matrix((np.array(vals, np.float32), (rows, cols)), shape=(len(qs), len(self.vocab)))

    def scores(self, qs, k1, b):
        tf = self.tf.tocoo()
        norm = k1 * (1 - b + b * self.dl / self.dl.mean())
        w = tf.data * (k1 + 1) / (tf.data + norm[tf.row]) * self.idf[tf.col]
        W = sp.csr_matrix((w.astype(np.float32), (tf.row, tf.col)), shape=tf.shape)
        key = id(qs)
        if key not in self.q:
            self.q[key] = self.query_matrix(qs)
        return np.asarray((self.q[key] @ W.T).todense(), dtype=np.float32)

def mask_self(scores, split):
    pos = {c: j for j, c in enumerate(POOL_IDS[split])}
    for i, q in enumerate(QUERIES[split]):
        j = pos.get(q["id"])
        if j is not None:
            scores[i, j] = -np.inf
    return scores

def topk_ids(scores, split, k=TOP_K):
    ids = POOL_IDS[split]
    k = min(k, scores.shape[1])
    part = np.argpartition(-scores, k - 1, axis=1)[:, :k]
    out = []
    for i in range(scores.shape[0]):
        top = part[i][np.argsort(-scores[i, part[i]])]
        out.append([ids[j] for j in top])
    return out

def retrieval_scores(ranked, split, k=TOP_K):
    tp = fp = fn = 0
    for q, r in zip(QUERIES[split], ranked):
        g = set(q["gold"]); r = r[:k]
        h = sum(c in g for c in r)
        tp += h; fp += len(r) - h; fn += len(g) - h
    P = tp / max(1, tp + fp); R = tp / max(1, tp + fn)
    return dict(micro_f1=2 * P * R / max(1e-12, P + R), micro_precision=P, micro_recall=R)

def recall_at(scores, split, k):
    hit = tot = 0
    for q, r in zip(QUERIES[split], topk_ids(scores, split, k)):
        g = set(q["gold"]); hit += len(g & set(r)); tot += len(g)
    return hit / max(1, tot)

POOL_IDS = {s: list(POOLS[s]) for s in SPLITS}
t0 = time.time()
bm25_grid, _idx_cache = [], {}
for ng in BM25_NGRAMS_GRID:
    _idx_cache[ng] = BM25Index("dev", ng)
    for k1 in BM25_K1_GRID:
        for b in BM25_B_GRID:
            sc = mask_self(_idx_cache[ng].scores(QUERIES["dev"], k1, b), "dev")
            bm25_grid.append(dict(ngrams="+".join(map(str, ng)), k1=k1, b=b, **retrieval_scores(topk_ids(sc, "dev"), "dev")))
_best = max(bm25_grid, key=lambda r: r["micro_f1"])
BM25_PARAMS = dict(ngrams=tuple(int(x) for x in _best["ngrams"].split("+")), k1=_best["k1"], b=_best["b"])
print(f"BM25 grid on dev ({time.time()-t0:.0f}s): best {BM25_PARAMS} -> Micro-F1@{TOP_K} {_best['micro_f1']:.4f} "
      f"(P {_best['micro_precision']:.4f}, R {_best['micro_recall']:.4f})")
BM25 = {}
for s in SPLITS:
    idx = _idx_cache[BM25_PARAMS["ngrams"]] if s == "dev" else BM25Index(s, BM25_PARAMS["ngrams"])
    BM25[s] = mask_self(idx.scores(QUERIES[s], BM25_PARAMS["k1"], BM25_PARAMS["b"]), s)
del _idx_cache
gc.collect()
for s in ("train", "dev"):
    print(f"BM25 {s}: Micro-F1@{TOP_K} {retrieval_scores(topk_ids(BM25[s], s), s)['micro_f1']:.4f} | "
          f"recall@50 {recall_at(BM25[s], s, 50):.3f} | recall@100 {recall_at(BM25[s], s, 100):.3f}")

HARD_NEG = {}
for q, r in zip(QUERIES["train"], topk_ids(BM25["train"], "train", HARD_NEG_POOL)):
    g = set(q["gold"])
    HARD_NEG[q["id"]] = [c for c in r if c not in g and c != q["id"]]

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = float(eps)

    def forward(self, x):
        dt = x.dtype
        xf = x.float()
        out = (xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)).to(dtype=dt)
        return out * self.weight.to(dtype=dt)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10_000.0):
        super().__init__()
        inv_freq = 1.0 / (float(base) ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        fr = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", fr.cos().repeat_interleave(2, dim=-1)[None, None], persistent=False)
        self.register_buffer("sin_cached", fr.sin().repeat_interleave(2, dim=-1)[None, None], persistent=False)

    @staticmethod
    def rotate_half(x):
        return torch.stack((-x[..., 1::2], x[..., 0::2]), dim=-1).flatten(-2)

    def forward(self, x):
        n = x.size(-2)
        cos = self.cos_cached[:, :, :n].to(device=x.device, dtype=x.dtype)
        sin = self.sin_cached[:, :, :n].to(device=x.device, dtype=x.dtype)
        return x * cos + self.rotate_half(x) * sin

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout, use_rope, rope_base):
        super().__init__()
        self.n_heads, self.n_kv_heads = int(n_heads), int(n_kv_heads)
        self.groups = self.n_heads // self.n_kv_heads
        self.head_dim = d_model // self.n_heads
        self.dropout = float(dropout)
        self.use_rope = bool(use_rope)
        kv = self.n_kv_heads * self.head_dim
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, kv)
        self.v_proj = nn.Linear(d_model, kv)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len, rope_base) if self.use_rope else None

    def forward(self, x, kv_states=None, key_padding_mask=None, causal=False):
        b, t, w = x.shape
        kvs = x if kv_states is None else kv_states
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(kvs).view(b, kvs.size(1), self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(kvs).view(b, kvs.size(1), self.n_kv_heads, self.head_dim).transpose(1, 2)
        if self.use_rope:
            q, k = self.rope(q), self.rope(k)
        k = k.repeat_interleave(self.groups, dim=1)
        v = v.repeat_interleave(self.groups, dim=1)
        allowed = None if key_padding_mask is None else key_padding_mask[:, None, None, :].bool()
        if causal:
            c = torch.ones(t, k.size(-2), dtype=torch.bool, device=x.device).tril()[None, None]
            allowed = c if allowed is None else allowed & c
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(y.transpose(1, 2).contiguous().view(b, t, w))

class FeedForward(nn.Module):
    def __init__(self, d_model, hidden, dropout, swiglu):
        super().__init__()
        self.swiglu = bool(swiglu)
        if self.swiglu:
            self.gate_proj = nn.Linear(d_model, hidden, bias=False)
            self.up_proj = nn.Linear(d_model, hidden, bias=False)
            self.down_proj = nn.Linear(hidden, d_model, bias=False)
            self.drop = nn.Dropout(dropout)
        else:
            self.net = nn.Sequential(nn.Linear(d_model, hidden), nn.GELU(), nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x):
        if self.swiglu:
            return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d, h, kvh, ff, ctx, dropout, swiglu, rope_base, eps):
        super().__init__()
        self.norm_attn, self.norm_ff = RMSNorm(d, eps), RMSNorm(d, eps)
        self.self_attn = GroupedQueryAttention(d, h, kvh, ctx, dropout, True, rope_base)
        self.ff = FeedForward(d, ff, dropout, swiglu)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = x + self.resid_dropout(self.self_attn(self.norm_attn(x), key_padding_mask=mask, causal=False))
        return x + self.resid_dropout(self.ff(self.norm_ff(x)))

class DecoderLayer(nn.Module):
    def __init__(self, d, h, kvh, ff, ctx, dropout, swiglu, rope_base, eps):
        super().__init__()
        self.norm_self, self.norm_cross, self.norm_ff = RMSNorm(d, eps), RMSNorm(d, eps), RMSNorm(d, eps)
        self.self_attn = GroupedQueryAttention(d, h, kvh, ctx, dropout, True, rope_base)
        self.cross_attn = GroupedQueryAttention(d, h, kvh, ctx, dropout, False, rope_base)
        self.ff = FeedForward(d, ff, dropout, swiglu)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, memory, mem_mask, dec_mask):
        x = x + self.resid_dropout(self.self_attn(self.norm_self(x), key_padding_mask=dec_mask, causal=True))
        x = x + self.resid_dropout(self.cross_attn(self.norm_cross(x), kv_states=memory, key_padding_mask=mem_mask))
        return x + self.resid_dropout(self.ff(self.norm_ff(x)))

class EncDecBackbone(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_kv_heads, n_enc, n_dec, ff_hidden, context, dropout,
                 swiglu=True, rope_base=10_000.0, eps=1e-6):
        super().__init__()
        self.d_model = d_model
        self.out_dim = 2 * d_model
        self.shared_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        args = (d_model, n_heads, n_kv_heads, ff_hidden, context, dropout, swiglu, rope_base, eps)
        self.encoder_layers = nn.ModuleList([EncoderLayer(*args) for _ in range(n_enc)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(*args) for _ in range(n_dec)])
        self.encoder_norm = RMSNorm(d_model, eps)
        self.decoder_norm = RMSNorm(d_model, eps)

    @property
    def blocks(self):
        return list(self.encoder_layers) + list(self.decoder_layers)

    def final_norms(self):
        return [self.encoder_norm, self.decoder_norm]

    def encode(self, src, mask, return_hidden=False, grad_ckpt=False):
        x = self.shared_embedding(src)
        hs = []
        for layer in self.encoder_layers:
            x = checkpoint(layer, x, mask, use_reentrant=False) if grad_ckpt else layer(x, mask)
            if return_hidden:
                hs.append(x)
        return self.encoder_norm(x), hs

    def decode(self, dec, memory, mem_mask, dec_mask, return_hidden=False, grad_ckpt=False):
        x = self.shared_embedding(dec)
        hs = []
        for layer in self.decoder_layers:
            if grad_ckpt:
                x = checkpoint(layer, x, memory, mem_mask, dec_mask, use_reentrant=False)
            else:
                x = layer(x, memory, mem_mask, dec_mask)
            if return_hidden:
                hs.append(x)
        return self.decoder_norm(x), hs

    @staticmethod
    def wrap(input_ids, attn_mask):
        b = input_ids.size(0)
        lengths = attn_mask.sum(1)
        bos = torch.full((b, 1), BOS_ID, dtype=input_ids.dtype, device=input_ids.device)
        pad = torch.full((b, 1), PAD_ID, dtype=input_ids.dtype, device=input_ids.device)
        t_ = torch.ones((b, 1), dtype=torch.bool, device=input_ids.device)
        f_ = torch.zeros((b, 1), dtype=torch.bool, device=input_ids.device)
        enc = torch.cat([bos, input_ids.masked_fill(~attn_mask, PAD_ID), pad], 1)
        enc_mask = torch.cat([t_, attn_mask, f_], 1)
        rows = torch.arange(b, device=input_ids.device)
        enc[rows, lengths + 1] = EOS_ID
        enc_mask[rows, lengths + 1] = True
        dec = torch.cat([bos, input_ids.masked_fill(~attn_mask, PAD_ID)], 1)
        dec_mask = torch.cat([t_, attn_mask], 1)
        return enc, enc_mask, dec, dec_mask

    def forward(self, input_ids, attn_mask, return_hidden=False, grad_ckpt=False):
        enc, enc_mask, dec, dec_mask = self.wrap(input_ids, attn_mask)
        mem, enc_hs = self.encode(enc, enc_mask, return_hidden, grad_ckpt)
        dh, dec_hs = self.decode(dec, mem, enc_mask, dec_mask, return_hidden, grad_ckpt)
        T = input_ids.size(1)
        h = torch.cat([mem[:, 1:T + 1], dh[:, 1:T + 1]], dim=-1)
        if return_hidden:
            return h, [x[:, 1:T + 1] for x in enc_hs + dec_hs]
        return h

def build_backbone(n_enc=None, n_dec=None, enc_map=None, dec_map=None, dropout=None):
    c = P1_CFG
    n_enc = n_enc or c["encoder_layers"]
    n_dec = n_dec or c["decoder_layers"]
    enc_map = enc_map or list(range(n_enc))
    dec_map = dec_map or list(range(n_dec))
    bb = EncDecBackbone(VOCAB_SIZE, c["d_model"], c["query_heads"], c["kv_heads"], n_enc, n_dec, c["ff_hidden"],
                        c["encoder_max_length"], BACKBONE_DROPOUT if dropout is None else dropout,
                        swiglu=(c.get("feed_forward", "swiglu") == "swiglu"), rope_base=c.get("rope_base", 10_000.0),
                        eps=c.get("rms_norm_eps", 1e-6))
    sd = {k: v for k, v in P1_STATE.items() if not k.startswith(("lm_head", "copy_head"))}
    new_sd = {k: v for k, v in sd.items() if not k.startswith(("encoder_layers.", "decoder_layers."))}
    for prefix, mapping in (("encoder_layers", enc_map), ("decoder_layers", dec_map)):
        for new_i, old_i in enumerate(mapping):
            pre = f"{prefix}.{old_i}."
            for k, v in sd.items():
                if k.startswith(pre):
                    new_sd[f"{prefix}.{new_i}." + k[len(pre):]] = v
    missing, unexpected = bb.load_state_dict(new_sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"Phase-1 checkpoint does not match backbone. missing={missing[:5]} unexpected={unexpected[:5]}")
    return bb

def build_student_backbone(cfg, drop):
    E, D = P1_CFG["encoder_layers"], P1_CFG["decoder_layers"]
    ne, nd = cfg.get("student_encoder_layers") or E, cfg.get("student_decoder_layers") or D
    em = [int(round(x)) for x in np.linspace(0, E - 1, ne)]
    dm = [int(round(x)) for x in np.linspace(0, D - 1, nd)]
    return build_backbone(n_enc=ne, n_dec=nd, enc_map=em, dec_map=dm, dropout=drop), em + [E + j for j in dm]

class TokenEncoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone, self.out_dim = backbone, backbone.out_dim

    def forward(self, ids, mask, mem_ids=None, mem_mask=None, starts=None, return_hidden=False, grad_ckpt=False):
        return self.backbone(ids, mask, return_hidden=return_hidden, grad_ckpt=grad_ckpt)

_probe = build_backbone()
P1_PARAMS = sum(p.numel() for p in _probe.parameters())
print(f"Phase-1 encoder-decoder used for tagging: {P1_PARAMS/1e6:.2f}M parameters (LM/copy heads excluded)")
assert MIN_PARAMS <= P1_PARAMS <= MAX_PARAMS or SMOKE_TEST
del _probe

In [ ]:
class LoRMAPlusLinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias

NF4_LEVELS = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
                           -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
                           0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
                           0.7229568362236023, 1.0])
QBLOCK = 64

def nf4_quantize(w):
    flat = w.detach().float().reshape(-1)
    pad = (-flat.numel()) % QBLOCK
    flat = torch.cat([flat, flat.new_zeros(pad)])
    blocks = flat.view(-1, QBLOCK)
    absmax = blocks.abs().max(1).values.clamp(min=1e-8)
    codes = (blocks / absmax[:, None]).unsqueeze(-1).sub(NF4_LEVELS).abs().argmin(-1).to(torch.uint8).view(-1)
    packed = (codes[0::2] | (codes[1::2] << 4)).contiguous()
    return packed, absmax.half(), pad

class QLoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.out_features, self.in_features = base.out_features, base.in_features
        packed, absmax, pad = nf4_quantize(base.weight)
        self.register_buffer("qweight", packed)
        self.register_buffer("absmax", absmax)
        self.register_buffer("levels", NF4_LEVELS.clone())
        self.pad = pad
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, self.in_features))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def dequantize(self, dtype):
        codes = torch.stack([self.qweight & 15, self.qweight >> 4], 1).view(-1).long()
        w = (self.levels[codes].view(-1, QBLOCK) * self.absmax.float()[:, None]).view(-1)
        if self.pad:
            w = w[:-self.pad]
        return w.view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):

        y = F.linear(x, self.dequantize(torch.float32), self.bias)
        return y + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)

class IA3Out(nn.Module):
    def __init__(self, base, *args, **kw):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l.to(x.dtype)

class IA3In(nn.Module):
    def __init__(self, base, *args, **kw):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l.to(x.dtype))

class AdaLoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r = base, r
        self.scale = alpha / r
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.E = nn.Parameter(torch.zeros(r))
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.Q) * self.E.to(x.dtype), self.P)

    def orth_penalty(self):
        I = torch.eye(self.r, device=self.Q.device)
        return (torch.norm(self.Q @ self.Q.T - I, p="fro") + torch.norm(self.P.T @ self.P - I, p="fro"))

class RankAllocator:
    def __init__(self, modules, cfg, total_steps):
        self.mods, self.cfg, self.total = modules, cfg, total_steps
        self.b1, self.b2 = cfg["beta1"], cfg["beta2"]
        self.init_budget = sum(m.r for m in modules)
        self.target_budget = cfg["target_rank"] * len(modules)
        self.ipt, self.unc = {}, {}
        self.final_keep = None
        self.budget = self.init_budget

    @torch.no_grad()
    def update_importance(self):
        for mi, m in enumerate(self.mods):
            for name in ("P", "E", "Q"):
                p = getattr(m, name)
                if p.grad is None:
                    continue
                cur = (p * p.grad).abs().float()
                key = (mi, name)
                if key not in self.ipt:
                    self.ipt[key], self.unc[key] = torch.zeros_like(cur), torch.zeros_like(cur)
                self.ipt[key].mul_(self.b1).add_((1 - self.b1) * cur)
                self.unc[key].mul_(self.b2).add_((1 - self.b2) * (cur - self.ipt[key]).abs())

    def _scores(self):
        out = []
        for mi, m in enumerate(self.mods):
            s = lambda n: self.ipt.get((mi, n), torch.zeros(1, device=m.E.device)) * \
                self.unc.get((mi, n), torch.zeros(1, device=m.E.device))
            sE = s("E").reshape(-1) if (mi, "E") in self.ipt else torch.zeros(m.r, device=m.E.device)
            sP = s("P").mean(0) if (mi, "P") in self.ipt else torch.zeros(m.r, device=m.E.device)
            sQ = s("Q").mean(1) if (mi, "Q") in self.ipt else torch.zeros(m.r, device=m.E.device)
            out.append(sE + sP + sQ)
        return out

    @torch.no_grad()
    def _apply(self, keep):
        for m, k in zip(self.mods, keep):
            m.E.masked_fill_(~k, 0.0)

    @torch.no_grad()
    def step(self, step):
        c, T = self.cfg, self.total
        t_init, t_final = int(c["t_init"] * T), int(c["t_final"] * T)
        if step <= t_init:
            return
        if step > T - t_final:
            if self.final_keep is None:
                self.final_keep = self._keep(self.target_budget)
            self._apply(self.final_keep)
            self.budget = self.target_budget
            return
        frac = 1 - (step - t_init) / max(1, T - t_final - t_init)
        self.budget = int((self.init_budget - self.target_budget) * frac ** 3 + self.target_budget)
        if step % c["delta_t"] == 0:
            self._apply(self._keep(self.budget))

    def _keep(self, budget):
        scores = self._scores()
        flat = torch.cat(scores)
        if budget >= flat.numel():
            return [torch.ones_like(s, dtype=torch.bool) for s in scores]
        keep = torch.zeros(flat.numel(), dtype=torch.bool, device=flat.device)
        keep[torch.topk(flat, budget).indices] = True
        return list(torch.split(keep, [s.numel() for s in scores]))

    def ranks(self):
        return [int((m.E != 0).sum()) for m in self.mods]

class HoulsbyAdapter(nn.Module):
    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))

ADAPTED_TYPES = (LoRMAPlusLinear, QLoRALinear, AdaLoRALinear, HoulsbyAdapter, IA3Out, IA3In)

def _resolve(block, path):
    parent = block
    parts = path.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1], getattr(parent, parts[-1])

def wrap_linears(backbone, targets, factory):
    n = 0
    for block in backbone.blocks:
        for path in targets:
            try:
                parent, name, leaf = _resolve(block, path)
            except AttributeError:
                continue
            if isinstance(leaf, nn.Linear):
                setattr(parent, name, factory(leaf)); n += 1
    return n

def is_norm(m):
    return isinstance(m, nn.LayerNorm) or "RMSNorm" in type(m).__name__

def block_param_ids(bb):
    return [{id(p) for p in b.parameters()} for b in bb.blocks]

def top_extras(bb):
    inside = set().union(*block_param_ids(bb))
    return [p for n, p in bb.named_parameters() if id(p) not in inside and "emb" not in n.split(".")[0]]

def configure_trainable(model, cfg):
    method = cfg["method"]
    bb = model.encoder.backbone
    for p in model.parameters():
        p.requires_grad = False
    save = set()
    if method == "frozen":
        pass
    elif method in ("full", "distill"):
        for p in model.parameters():
            p.requires_grad = True
    elif method == "lorma_plus":
        for m in bb.modules():
            if isinstance(m, LoRMAPlusLinear):
                m.A.requires_grad = m.B.requires_grad = True
    elif method == "adalora":
        for m in bb.modules():
            if isinstance(m, AdaLoRALinear):
                m.P.requires_grad = m.E.requires_grad = m.Q.requires_grad = True
    elif method == "ia3":
        for m in bb.modules():
            if isinstance(m, (IA3Out, IA3In)):
                m.l.requires_grad = True
    elif method == "qlora":
        for m in bb.modules():
            if isinstance(m, QLoRALinear):
                m.A.requires_grad = m.B.requires_grad = True
    elif method == "prefix":
        for p in bb.prefix.parameters():
            p.requires_grad = True
    elif method == "adapter":
        for m in bb.modules():
            if isinstance(m, HoulsbyAdapter):
                for p in list(m.down.parameters()) + list(m.up.parameters()):
                    p.requires_grad = True
            elif is_norm(m):
                for p in m.parameters():
                    p.requires_grad = True
    elif method == "partial":
        for b in list(bb.blocks)[-cfg["top_layers"]:]:
            for p in b.parameters():
                p.requires_grad = True
        for p in top_extras(bb):
            p.requires_grad = True
    else:
        raise ValueError(method)
    for m in model.head_modules():
        for p in m.parameters():
            p.requires_grad = True
    save |= {n for n, p in model.named_parameters() if p.requires_grad}
    model.save_names = save

def param_groups(model, cfg):
    head_ids = {id(p) for m in model.head_modules() for p in m.parameters()}
    head = [p for p in model.parameters() if id(p) in head_ids]
    groups = [dict(params=head, lr=cfg["head_lr"], weight_decay=0.0)]
    decay = WEIGHT_DECAY if cfg["method"] in ("full", "distill", "partial") else 0.0
    bb = model.encoder.backbone
    if cfg["method"] == "partial":
        blocks = list(bb.blocks)[::-1]
        k = cfg["top_layers"]
        for depth, b in enumerate(blocks[:k]):
            groups.append(dict(params=list(b.parameters()), lr=cfg["lr"] * cfg["llrd"] ** depth, weight_decay=decay))
        groups.append(dict(params=top_extras(bb), lr=cfg["lr"], weight_decay=0.0))
    else:
        body = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
        if body:
            groups.append(dict(params=body, lr=cfg["lr"], weight_decay=decay))
    for g in groups:
        g["initial_lr"] = g["lr"]
    return [g for g in groups if g["params"]]

def make_controller(model, cfg, total_steps):
    if cfg["method"] == "adalora":
        mods = [m for m in model.encoder.backbone.modules() if isinstance(m, AdaLoRALinear)]
        return RankAllocator(mods, cfg, total_steps)
    return None

def extra_loss(model, cfg):
    if cfg["method"] != "adalora":
        return None
    mods = [m for m in model.encoder.backbone.modules() if isinstance(m, AdaLoRALinear)]
    return cfg["orth_reg"] * torch.stack([m.orth_penalty() for m in mods]).mean()

def build_retriever(cfg):
    method = cfg["method"]
    drop = cfg["dropout"] if cfg.get("dropout") is not None else BACKBONE_DROPOUT
    if method == "distill":
        backbone, lm = build_student_backbone(cfg, drop)
        cfg["layer_map"] = lm
    else:
        backbone = build_backbone(dropout=drop)
    if method == "lorma_plus":
        n = wrap_linears(backbone, LOWRANK_TARGETS, lambda l: LoRMAPlusLinear(l, cfg["rank"], cfg["alpha"]))
        print(f"lorma_plus: adapted {n} linear layers")
    elif method == "adalora":
        n = wrap_linears(backbone, LOWRANK_TARGETS, lambda l: AdaLoRALinear(l, cfg["init_rank"], cfg["alpha"]))
        print(f"adalora: adapted {n} linear layers (rank {cfg['init_rank']} -> avg {cfg['target_rank']})")
    elif method == "adapter":
        n = wrap_linears(backbone, ADAPTER_TARGETS, lambda l: HoulsbyAdapter(l, cfg["bottleneck"]))
        print(f"adapter: {n} Houlsby adapters (bottleneck {cfg['bottleneck']})")
    elif method == "ia3":
        n = 0
        for path, where in IA3_TARGETS.items():
            n += wrap_linears(backbone, [path], (lambda l: IA3Out(l)) if where == "out" else (lambda l: IA3In(l)))
        print(f"ia3: {n} rescaling vectors (keys, values, FFN activations)")
    elif method == "qlora":
        n = wrap_linears(backbone, LOWRANK_TARGETS, lambda l: QLoRALinear(l, cfg["rank"], cfg["alpha"]))
        print(f"qlora: {n} linear layers stored in 4-bit NF4 + LoRA rank {cfg['rank']}")
    elif method == "prefix":
        if not hasattr(backbone, "add_prefix"):
            raise ValueError("prefix tuning is implemented for the multi-resolution memory model only")
        backbone.add_prefix(cfg["prefix_len"], cfg["prefix_hidden"])
        print(f"prefix: {cfg['prefix_len']} learned prefix states in each of {len(backbone.blocks)} layers")
    cfg["selection"] = cfg.get("selection") or SELECTION
    cfg["hard_negs"] = HARD_NEGS if cfg.get("hard_negs") is None else cfg["hard_negs"]
    model = Retriever(TokenEncoder(backbone))
    model.cfg_selection = cfg["selection"]
    model.grad_ckpt = USE_GRAD_CHECKPOINTING
    configure_trainable(model, cfg)
    return model

def _adapter_checks():
    torch.manual_seed(0)
    base = nn.Linear(24, 32)
    x = torch.randn(3, 24)
    ref = base(x)
    for name, m in [("lorma_plus", LoRMAPlusLinear(base, 4, 4)), ("adalora", AdaLoRALinear(base, 4, 8)),
                    ("houlsby", HoulsbyAdapter(base, 8))]:
        assert torch.allclose(m(x), ref, atol=1e-5), f"{name} not identity at init"
    for m in (IA3Out(base), IA3In(base)):
        assert torch.allclose(m(x), ref, atol=1e-5), "IA3 not identity at init"
    q = QLoRALinear(base, 4, 8)
    rel = (q.dequantize(torch.float32) - base.weight).norm() / base.weight.norm()
    assert rel < 0.15, f"NF4 reconstruction error too high: {rel:.3f}"

    mods = [AdaLoRALinear(nn.Linear(16, 16), 6, 8) for _ in range(3)]
    for m in mods:
        m.E.data.uniform_(0.5, 1.0)
        for p in (m.P, m.E, m.Q):
            p.grad = torch.randn_like(p)
    ra = RankAllocator(mods, dict(beta1=0.85, beta2=0.85, target_rank=2, t_init=0.0, t_final=0.0, delta_t=1), 10)
    ra.update_importance()
    ra._apply(ra._keep(7))
    assert sum(ra.ranks()) == 7, ra.ranks()
    print(f"Adapter checks passed (LoRMA+, AdaLoRA, Houlsby = identity at init; AdaLoRA budget masking works; "
          f"NF4 4-bit relative weight error {rel:.3f}).")

_adapter_checks()

In [ ]:
class Retriever(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        d_in = encoder.out_dim * (2 if CHUNK_POOLING == "mean+last" else 1)
        self.proj = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(d_in, EMB_DIM))
        self.pos_emb = nn.Embedding(N_POS_BUCKETS, EMB_DIM)
        nn.init.zeros_(self.pos_emb.weight)
        self.scale = nn.Parameter(torch.tensor(math.log(1 / INIT_TEMPERATURE)))
        self.grad_ckpt = False
        self.save_names = set()

    def head_modules(self):
        holder = nn.Module(); holder.scale = self.scale
        return [self.proj, self.pos_emb, holder]

    def chunk_embeddings(self, b):
        h = self.encoder(b["ids"], b["mask"], b["mem_ids"], b["mem_mask"], b["starts"],
                         grad_ckpt=self.grad_ckpt and self.training).float()
        m = b["mask"].float()
        lens = m.sum(1).clamp(min=1)
        v = (h * m[..., None]).sum(1) / lens[:, None]
        if CHUNK_POOLING == "mean+last":
            v = torch.cat([v, h[torch.arange(h.size(0), device=h.device), lens.long() - 1]], -1)
        return F.normalize(self.proj(v).float() + self.pos_emb(b["c_bucket"]).float(), dim=-1)

    def forward(self, b):
        ce = self.chunk_embeddings(b)
        B = b["n_chunks"].numel()
        doc = torch.zeros(B, ce.size(-1), device=ce.device).index_add_(0, b["c_doc"], ce)
        return F.normalize(doc / b["n_chunks"][:, None].float(), dim=-1), ce

@torch.no_grad()
def encode_docs(model, docs, K=None, mode=None):
    model.eval()
    K, mode = K or EVAL_N_CHUNKS, mode or model.cfg_selection
    bs = max(1, EVAL_BATCH_DOCS * N_CHUNKS // K)
    E, C, M = [], [], []
    for i in tqdm(range(0, len(docs), bs), desc=f"encoding {len(docs)} cases", leave=False):
        items = [make_doc_item(d, mode, K) for d in docs[i:i + bs]]
        b = to_device(collate_docs(items))
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            de, ce = model(b)
        grid = torch.zeros(len(items), K, ce.size(-1), device=ce.device)
        msk = torch.zeros(len(items), K, dtype=torch.bool, device=ce.device)
        grid[b["c_doc"], b["c_slot"]] = ce.float(); msk[b["c_doc"], b["c_slot"]] = True
        E.append(de.float().cpu()); C.append(grid.half().cpu()); M.append(msk.cpu())
    return torch.cat(E), torch.cat(C), torch.cat(M)

def encode_split(model, split, K=None, mode=None):
    qd = [DOCS[(split, "q", q["id"])] for q in QUERIES[split]]
    cd = [DOCS[(split, "c", c)] for c in POOL_IDS[split]]
    return encode_docs(model, qd, K, mode), encode_docs(model, cd, K, mode)

@torch.no_grad()
def dense_scores(q_enc, c_enc, kind="cosine"):
    qE, qC, qM = q_enc; cE, cC, cM = c_enc
    if kind == "cosine":
        return (qE @ cE.T).numpy()
    dev = DEVICE
    cC_d, cM_d = cC.to(dev).float(), cM.to(dev)
    out = np.zeros((qE.size(0), cE.size(0)), np.float32)
    for i in range(0, qE.size(0), 16):
        q, qm = qC[i:i + 16].to(dev).float(), qM[i:i + 16].to(dev)
        for j in range(0, cE.size(0), 512):
            s = torch.einsum("qad,cbd->qcab", q, cC_d[j:j + 512])
            s = s.masked_fill(~cM_d[j:j + 512][None, :, None, :], -1e4).max(-1).values
            s = (s * qm[:, None, :]).sum(-1) / qm.sum(-1).clamp(min=1)[:, None]
            out[i:i + 16, j:j + 512] = s.cpu().numpy()
    return out

def znorm(s):
    s = s.copy()
    fin = np.isfinite(s)
    mu = np.where(fin, s, 0).sum(1, keepdims=True) / fin.sum(1, keepdims=True).clip(1)
    sd = np.sqrt((np.where(fin, s - mu, 0) ** 2).sum(1, keepdims=True) / fin.sum(1, keepdims=True).clip(1)) + 1e-6
    return np.where(fin, (s - mu) / sd, -np.inf)

def rrf(*mats, k=60):
    out = np.zeros_like(mats[0])
    for m in mats:
        r = np.argsort(np.argsort(-m, 1), 1)
        out += 1.0 / (k + 1 + r)
    return out

FUSION_W = np.round(np.arange(0.0, 1.0001, 0.1), 2)

def tune_fusion(dense_dev, split="dev"):
    zd, zb = znorm(mask_self(dense_dev.copy(), split)), znorm(BM25[split])
    rows = [dict(fusion=f"z-score w={w}", w=float(w),
                 **retrieval_scores(topk_ids(np.nan_to_num(w * zd + (1 - w) * zb, neginf=-1e9), split), split))
            for w in FUSION_W]
    rows.append(dict(fusion="RRF", w=None, **retrieval_scores(topk_ids(rrf(zd, zb), split), split)))
    return rows

def fuse(dense, split, best):
    zd, zb = znorm(mask_self(dense.copy(), split)), znorm(BM25[split])
    s = rrf(zd, zb) if best["w"] is None else best["w"] * zd + (1 - best["w"]) * zb
    return mask_self(np.nan_to_num(s, neginf=-1e9), split)

def write_submission(run_dir, ranked):
    pool = set(POOL_IDS["test"])
    sub = {q["id"]: r[:TOP_K] for q, r in zip(QUERIES["test"], ranked)}
    assert list(sub) == test_ids
    for qid, r in sub.items():
        assert len(r) == TOP_K and len(set(r)) == TOP_K and set(r) <= pool and qid not in r, qid
    pj = Path(run_dir) / "prediction.json"
    pj.write_text(json.dumps(sub, ensure_ascii=False, indent=1), encoding="utf-8")
    zp = Path(run_dir) / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    return zp

_bm_dir = OUTPUT_DIR / "runs" / "BM25_baseline"; _bm_dir.mkdir(parents=True, exist_ok=True)
BM25_DEV = retrieval_scores(topk_ids(BM25["dev"], "dev"), "dev")
if WRITE_SUBMISSIONS:
    write_submission(_bm_dir, topk_ids(BM25["test"], "test"))
save_json(_bm_dir / "results.json", dict(name="BM25_baseline", params=BM25_PARAMS, dev=BM25_DEV))

In [ ]:
TRAINED_MODELS, RESULTS = {}, {}

def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))

def logical_params(module):
    q = sum(m.out_features * m.in_features for m in module.modules() if isinstance(m, QLoRALinear))
    return count_params(module) + q

def make_cfg(name):
    cfg = dict(EXPERIMENT_DEFAULTS)
    cfg.update(copy.deepcopy(EXPERIMENTS[name]))
    cfg["name"] = name
    cfg["epochs"] = cfg.get("epochs") or EPOCHS
    return cfg

def saved_state(model):
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in model.save_names}

def load_trained(name):
    run_dir = OUTPUT_DIR / "runs" / name
    cfg = json.loads((run_dir / "cfg.json").read_text())
    model = build_retriever(cfg)
    model.load_state_dict(torch.load(run_dir / "best_state.pt", map_location="cpu"), strict=False)
    return model

TRAIN_PAIRS = [(q["id"], c) for q in QUERIES["train"] for c in q["gold"] if c in POOLS["train"] and c != q["id"]]
TRAIN_GOLD = {q["id"]: set(q["gold"]) for q in QUERIES["train"]}
print(f"{len(TRAIN_PAIRS):,} (query, gold candidate) training pairs from {len(QUERIES['train']):,} queries")

def make_batch(pairs, H, rng, K, mode):
    qids = [q for q, _ in pairs]
    cands = [c for _, c in pairs]
    for q in qids:
        pool = [c for c in HARD_NEG.get(q, []) if c not in cands]
        cands += rng.sample(pool, min(H, len(pool)))
    items = [make_doc_item(DOCS[("train", "q", q)], mode, K) for q in qids] + \
            [make_doc_item(DOCS[("train", "c", c)], mode, K) for c in cands]
    valid = torch.ones(len(qids), len(cands), dtype=torch.bool)
    for i, q in enumerate(qids):
        for j, c in enumerate(cands):
            if j != i and (c in TRAIN_GOLD[q] or c == q):
                valid[i, j] = False
    return collate_docs(items), len(qids), valid

def score_matrix(model, b, nq, valid):
    de, _ = model(b)
    S = model.scale.exp().clamp(max=100) * de[:nq] @ de[nq:].T
    return S.masked_fill(~valid, -1e4), de

@torch.no_grad()
def dev_eval(model, K=None):
    (qe, cE) = encode_split(model, "dev", K or N_CHUNKS)
    dense = dense_scores(qe, cE, "cosine")
    rows = tune_fusion(dense)
    dense_only = next(r for r in rows if r["w"] == 1.0)
    best = max(rows, key=lambda r: r["micro_f1"])
    return best["micro_f1"], dense_only["micro_f1"], best

def run_experiment(name, teacher=None):
    cfg = make_cfg(name)
    run_dir = OUTPUT_DIR / "runs" / name
    run_dir.mkdir(parents=True, exist_ok=True)
    if RESUME_COMPLETED_RUNS and (run_dir / "results.json").exists():
        RESULTS[name] = json.loads((run_dir / "results.json").read_text())
        print(f"[{name}] already finished (dev Micro-F1@{TOP_K} {RESULTS[name]['dev']['micro_f1']:.4f}) - skipping")
        return RESULTS[name]
    print("=" * 100); print(f"{name}: {cfg['description']}"); print("=" * 100)
    seed_everything(SEED)
    is_kd = cfg["method"] == "distill"
    model = build_retriever(cfg).to(DEVICE)
    n_bb, n_total = logical_params(model.encoder), logical_params(model)
    n_train = sum(p.numel() for n, p in model.named_parameters() if n in model.save_names)
    print(f"backbone (+adapters) {n_bb/1e6:.2f}M | trainable {n_train/1e6:.3f}M ({100*n_train/n_total:.2f}%) | "
          f"{cfg['selection']} {N_CHUNKS}x{CHUNK_TOKENS} (train) / {EVAL_N_CHUNKS}x{CHUNK_TOKENS} (pools) | "
          f"hard negatives/query {cfg['hard_negs']}")
    assert MIN_PARAMS <= n_bb <= MAX_PARAMS or SMOKE_TEST, \
        f"backbone {n_bb/1e6:.1f}M outside the {MIN_PARAMS/1e6:.0f}M-{MAX_PARAMS/1e6:.0f}M limit"
    if is_kd:
        assert teacher is not None, "distillation needs a teacher"
        teacher.to(DEVICE).eval()
        for p in teacher.parameters():
            p.requires_grad = False
        print(f"teacher {logical_params(teacher.encoder)/1e6:.1f}M -> student {n_bb/1e6:.1f}M (layers {cfg['layer_map']})")
    groups = param_groups(model, cfg)
    optim = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8)
    steps_per_epoch = max(1, math.ceil(len(TRAIN_PAIRS) / (PAIRS_PER_BATCH * GRAD_ACCUM)))
    total = steps_per_epoch * cfg["epochs"]
    warm = max(1, int(WARMUP_FRACTION * total))
    sched = torch.optim.lr_scheduler.LambdaLR(optim, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    controller = make_controller(model, cfg, total)
    use_amp = cfg["use_amp"] and DEVICE.type == "cuda"
    scaler = make_grad_scaler(use_amp)
    params = [p for g in groups for p in g["params"]]
    print(f"{len(TRAIN_PAIRS):,} pairs x {cfg['epochs']} epochs | optimizer steps {total} | budget {MAX_TRAIN_MINUTES} min")
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    rng = random.Random(SEED)
    history, best, best_ep, bad, t0, gstep, out_of_time = [], -1.0, 0, 0, time.time(), 0, False

    def step():
        nonlocal gstep
        if scaler is not None:
            scaler.unscale_(optim)
        if controller is not None:
            controller.update_importance()
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
        if scaler is None:
            optim.step()
        else:
            scaler.step(optim); scaler.update()
        sched.step(); optim.zero_grad(set_to_none=True)
        gstep += 1
        if controller is not None:
            controller.step(gstep)

    for ep in range(1, cfg["epochs"] + 1):
        pairs = TRAIN_PAIRS[:]; rng.shuffle(pairs)
        model.train()
        run_loss, n_loss, micro, te = 0.0, 0, 0, time.time()
        n_batches = math.ceil(len(pairs) / PAIRS_PER_BATCH)
        optim.zero_grad(set_to_none=True)
        for bi in tqdm(range(n_batches), desc=f"{name} ep{ep}", leave=False):
            chunk = pairs[bi * PAIRS_PER_BATCH:(bi + 1) * PAIRS_PER_BATCH]
            b, nq, valid = make_batch(chunk, cfg["hard_negs"], rng, N_CHUNKS, cfg["selection"])
            b, valid = to_device(b), valid.to(DEVICE)
            target = torch.arange(nq, device=DEVICE)
            try:
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                    S, de = score_matrix(model, b, nq, valid)
                    loss = F.cross_entropy(S.float(), target)
                    if is_kd:
                        with torch.no_grad():
                            tS, tde = score_matrix(teacher, b, nq, valid)
                        T = cfg["kd_temperature"]
                        kl = F.kl_div(F.log_softmax(S.float() / T, -1), F.softmax(tS.float() / T, -1),
                                      reduction="batchmean") * T * T
                        emb = (de.float() - tde.float()).pow(2).sum(-1).mean()
                        loss = cfg["kd_alpha"] * loss + (1 - cfg["kd_alpha"]) * kl + cfg["kd_emb_weight"] * emb
                    reg = extra_loss(model, cfg)
                    if reg is not None:
                        loss = loss + reg
                if not torch.isfinite(loss):
                    optim.zero_grad(set_to_none=True); continue
                (loss / GRAD_ACCUM).backward() if scaler is None else scaler.scale(loss / GRAD_ACCUM).backward()
            except torch.cuda.OutOfMemoryError:
                optim.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                print("    CUDA out of memory on one batch -> skipped (lower PAIRS_PER_BATCH / HARD_NEGS if frequent)")
                continue
            micro += 1; run_loss += float(loss); n_loss += 1
            if micro % GRAD_ACCUM == 0:
                step()
            if micro == 20 * GRAD_ACCUM and ep == 1:
                print(f"    ~{(time.time() - te) / micro * n_batches / 60:.0f} min per epoch at the current speed")
            if time.time() - t0 > MAX_TRAIN_MINUTES * 60:
                out_of_time = True
                print(f"    time budget of {MAX_TRAIN_MINUTES} min reached in epoch {ep} ({micro}/{n_batches} batches)")
                break
        if micro % GRAD_ACCUM:
            step()
        f_fused, f_dense, bf = dev_eval(model)
        history.append(dict(epoch=ep, train_loss=run_loss / max(1, n_loss), dev_f1_fused=f_fused, dev_f1_dense=f_dense,
                            temperature=float(1 / model.scale.exp()), elapsed_s=time.time() - t0))
        print(f"  epoch {ep} | loss {history[-1]['train_loss']:.4f} | dev Micro-F1@{TOP_K}: dense {f_dense:.4f}, "
              f"fused {f_fused:.4f} ({bf['fusion']}) | temperature {history[-1]['temperature']:.3f} | {(time.time()-t0)/60:.1f} min")
        if f_fused > best + 1e-5:
            best, best_ep, bad = f_fused, ep, 0
            torch.save(saved_state(model), run_dir / "best_state.pt")
        else:
            bad += 1
            if bad >= cfg["patience"]:
                print(f"  early stop (no improvement for {cfg['patience']} epochs)")
                break
        if out_of_time:
            break
    train_time = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else 0.0
    model.load_state_dict(torch.load(run_dir / "best_state.pt", map_location=DEVICE), strict=False)

    te = time.time()
    dev_q, dev_c = encode_split(model, "dev")
    enc_s = (time.time() - te) / (len(QUERIES["dev"]) + len(POOL_IDS["dev"]))
    test_q, test_c = encode_split(model, "test")
    options = []
    for kind in ("cosine", "maxsim"):
        dd = dense_scores(dev_q, dev_c, kind)
        for r in tune_fusion(dd):
            options.append(dict(similarity=kind, **r))
        np.save(run_dir / f"dev_dense_{kind}.npy", dd.astype(np.float16))
        np.save(run_dir / f"test_dense_{kind}.npy", dense_scores(test_q, test_c, kind).astype(np.float16))
    choice = max(options, key=lambda r: r["micro_f1"])
    dense_only = {k: next(r for r in options if r["similarity"] == k and r["w"] == 1.0) for k in ("cosine", "maxsim")}
    dd = np.load(run_dir / f"dev_dense_{choice['similarity']}.npy").astype(np.float32)
    dev_final = fuse(dd, "dev", choice)
    print(f"[{name}] best epoch {best_ep} | dev Micro-F1@{TOP_K}: BM25 {BM25_DEV['micro_f1']:.4f} | dense cosine "
          f"{dense_only['cosine']['micro_f1']:.4f} | dense MaxSim {dense_only['maxsim']['micro_f1']:.4f} | best = "
          f"{choice['similarity']} + {choice['fusion']}: F1 {choice['micro_f1']:.4f} (P {choice['micro_precision']:.4f}, "
          f"R {choice['micro_recall']:.4f}) | {1000*enc_s:.0f} ms per case")
    out = dict(name=name, method=cfg["method"], description=cfg["description"], cfg=cfg,
               params_backbone=n_bb, params_total=n_total, params_trainable=n_train,
               trainable_pct=100 * n_train / n_total, train_time_s=train_time, best_epoch=best_ep,
               epochs_run=len(history), peak_mem_gb=peak, choice=choice,
               dev=dict(micro_f1=choice["micro_f1"], micro_precision=choice["micro_precision"],
                        micro_recall=choice["micro_recall"]),
               dev_dense_cosine=dense_only["cosine"], dev_dense_maxsim=dense_only["maxsim"], fusion_table=options,
               dev_dense_recall100=recall_at(mask_self(dd.copy(), "dev"), "dev", 100),
               ms_per_case=1000 * enc_s, index_mb=(len(POOL_IDS["test"]) * EMB_DIM * 2) / 1024**2,
               state_mb=(run_dir / "best_state.pt").stat().st_size / 1024**2, history=history)
    save_json(run_dir / "dev_ranked.json", dict(zip([q["id"] for q in QUERIES["dev"]], topk_ids(dev_final, "dev"))))
    if WRITE_SUBMISSIONS:
        td = np.load(run_dir / f"test_dense_{choice['similarity']}.npy").astype(np.float32)
        out["submission_zip"] = str(write_submission(run_dir, topk_ids(fuse(td, "test", choice), "test")))
        print(f"        submission: {out['submission_zip']}")
    save_json(run_dir / "cfg.json", cfg)
    save_json(run_dir / "results.json", out)
    RESULTS[name] = out
    model.cpu(); TRAINED_MODELS[name] = model
    if teacher is not None:
        teacher.cpu()
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return out

In [ ]:
for _p in sorted((OUTPUT_DIR / "runs").glob("*/results.json")):
    if _p.parent.name in EXPERIMENTS:
        RESULTS[_p.parent.name] = json.loads(_p.read_text())

TEACHER_CANDIDATES = [n for n, c in EXPERIMENTS.items() if c["method"] not in ("distill", "frozen")]

def pick_teacher():
    done = {n: RESULTS[n]["dev_dense_cosine"]["micro_f1"] for n in TEACHER_CANDIDATES if n in RESULTS}
    if not done:
        raise RuntimeError("Distillation needs a finished teacher: run the other experiments first.")
    name = max(done, key=done.get)
    return name, TRAINED_MODELS.get(name) or load_trained(name)

for name in RUN_LIST:
    teacher = None
    if EXPERIMENTS[name]["method"] == "distill" and not (OUTPUT_DIR / "runs" / name / "results.json").exists():
        tname, teacher = pick_teacher()
        EXPERIMENTS[name]["teacher_name"] = tname
        print(f"Teacher for {name}: {tname} (dev dense Micro-F1@{TOP_K} {RESULTS[tname]['dev_dense_cosine']['micro_f1']:.4f})")
    run_experiment(name, teacher)
    gc.collect()
print("\nFinished:", [n for n in RUN_LIST if n in RESULTS])

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
ORDER = [n for n in EXPERIMENTS if n in RESULTS]
rows = [dict(experiment="BM25 (lexical, no neural model)", method="bm25", trainable_pct=0.0, final_F1=BM25_DEV["micro_f1"],
             final_P=BM25_DEV["micro_precision"], final_R=BM25_DEV["micro_recall"], dense_cosine_F1=np.nan,
             dense_maxsim_F1=np.nan, chosen="-")]
for n in ORDER:
    r = RESULTS[n]
    rows.append(dict(experiment=n, method=r["method"], backbone_M=r["params_backbone"] / 1e6,
                     trainable_M=r["params_trainable"] / 1e6, trainable_pct=r["trainable_pct"],
                     final_F1=r["dev"]["micro_f1"], final_P=r["dev"]["micro_precision"], final_R=r["dev"]["micro_recall"],
                     dense_cosine_F1=r["dev_dense_cosine"]["micro_f1"], dense_maxsim_F1=r["dev_dense_maxsim"]["micro_f1"],
                     dense_recall100=r["dev_dense_recall100"], chosen=f"{r['choice']['similarity']} + {r['choice']['fusion']}",
                     train_min=r["train_time_s"] / 60, peak_mem_GB=r["peak_mem_gb"], ms_per_case=r["ms_per_case"],
                     ckpt_MB=r["state_mb"]))
summary = pd.DataFrame(rows).set_index("experiment")
summary.to_csv(OUTPUT_DIR / f"phase7_{ARCH_NAME}_results.csv")
print(f"Dev Micro-F1@{TOP_K} (final = the configuration used for the submission; dense = Phase-1 retriever alone)")
display(summary.round(4))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for n in ORDER:
    t = [r for r in RESULTS[n]["fusion_table"] if r["similarity"] == "cosine" and r["w"] is not None]
    ax[0].plot([r["w"] for r in t], [r["micro_f1"] for r in t], marker="o", label=n)
ax[0].set_xlabel("dense weight w (0 = BM25 only, 1 = dense only)"); ax[0].set_ylabel(f"dev Micro-F1@{TOP_K}")
ax[0].set_title("Lexical + dense fusion"); ax[0].legend(fontsize=7)
for n in ORDER:
    h = RESULTS[n]["history"]
    ax[1].plot([x["epoch"] for x in h], [x["dev_f1_dense"] for x in h], marker="o", label=n)
ax[1].set_title("Dense dev Micro-F1 per epoch"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "phase7_overview.png", dpi=130); plt.show()

best_name = max(ORDER, key=lambda n: RESULTS[n]["dev"]["micro_f1"])
best = RESULTS[best_name]
print(f"Best run: {best_name} ({best['choice']['similarity']} + {best['choice']['fusion']}, dev Micro-F1@{TOP_K} "
      f"{best['dev']['micro_f1']:.4f} vs BM25 {BM25_DEV['micro_f1']:.4f})")

ranked = json.loads((OUTPUT_DIR / "runs" / best_name / "dev_ranked.json").read_text())
bm_ranked = dict(zip([q["id"] for q in QUERIES["dev"]], topk_ids(BM25["dev"], "dev")))
dd = np.load(OUTPUT_DIR / "runs" / best_name / "dev_dense_cosine.npy").astype(np.float32)
dn_ranked = dict(zip([q["id"] for q in QUERIES["dev"]], topk_ids(mask_self(dd, "dev"), "dev")))
recs = []
for q in QUERIES["dev"]:
    g = set(q["gold"]); r = ranked[q["id"]]
    h = len(g & set(r))
    recs.append(dict(query=q["id"], gold=len(g), hits=h, f1=2 * h / (len(g) + TOP_K),
                     words=DOCS[("dev", "q", q["id"])]["n_words"], bm25_hits=len(g & set(bm_ranked[q["id"]])),
                     dense_hits=len(g & set(dn_ranked[q["id"]]))))
err = pd.DataFrame(recs)
print(f"\nqueries with no correct candidate in the top {TOP_K}: {np.mean(err.hits == 0):.1%}")
err["gold_bucket"] = pd.cut(err.gold, [0, 3, 7, 15, 10_000], labels=["1-3", "4-7", "8-15", "16+"])
display(err.groupby("gold_bucket", observed=True).agg(queries=("f1", "size"), mean_F1=("f1", "mean"),
                                                       mean_hits=("hits", "mean")).round(3))
err["length_quartile"] = pd.qcut(err.words, 4, labels=["Q1 short", "Q2", "Q3", "Q4 long"], duplicates="drop")
display(err.groupby("length_quartile", observed=True)[["f1", "bm25_hits", "dense_hits"]].mean().round(3))
both = (err.bm25_hits > 0) & (err.dense_hits > 0)
print(f"complementarity: queries where only BM25 finds a gold case {np.mean((err.bm25_hits > 0) & (err.dense_hits == 0)):.1%}, "
      f"only dense {np.mean((err.dense_hits > 0) & (err.bm25_hits == 0)):.1%}, both {np.mean(both):.1%}")
pos = {c: j for j, c in enumerate(POOL_IDS["dev"])}
fused = fuse(dd if best["choice"]["similarity"] == "cosine" else
             np.load(OUTPUT_DIR / "runs" / best_name / "dev_dense_maxsim.npy").astype(np.float32), "dev", best["choice"])
order = np.argsort(-fused, 1)
rank_of = np.empty_like(order); rank_of[np.arange(len(order))[:, None], order] = np.arange(order.shape[1])[None]
gold_ranks = [rank_of[i, pos[c]] + 1 for i, q in enumerate(QUERIES["dev"]) for c in q["gold"] if c in pos]
gr = np.array(gold_ranks)
print(f"rank of gold candidates in the final ranking: median {int(np.median(gr))} | in top 7: {np.mean(gr <= 7):.1%} | "
      f"top 20: {np.mean(gr <= 20):.1%} | top 100: {np.mean(gr <= 100):.1%} (pool {len(POOL_IDS['dev'])})")

if RUN_REPRESENTATION_STUDY:
    model = (TRAINED_MODELS.get(best_name) or load_trained(best_name)).to(DEVICE)
    rep = {}
    for mode, K in [("spread", N_CHUNKS), ("spread", EVAL_N_CHUNKS), ("head", EVAL_N_CHUNKS), ("head+tail", EVAL_N_CHUNKS)]:
        tq = time.time()
        qe, ce = encode_split(model, "dev", K, mode)
        s = dense_scores(qe, ce, "cosine")
        rep[f"{mode} x{K}"] = dict(tokens_per_case=K * CHUNK_TOKENS, dense_F1=retrieval_scores(topk_ids(mask_self(s, "dev"), "dev"), "dev")["micro_f1"],
                                   best_fused_F1=max(r["micro_f1"] for r in tune_fusion(s)), seconds=time.time() - tq)
    model.cpu()
    print(f"\nLong-document representation with {best_name} (dev, inference only):")
    display(pd.DataFrame(rep).T.round(4))

print("\n--- Key comparisons for the written analysis ---")
for n in ORDER:
    r = RESULTS[n]
    print(f"{n:30s}: dense {r['dev_dense_cosine']['micro_f1']:.4f} | final {r['dev']['micro_f1']:.4f} | "
          f"{r['trainable_pct']:.2f}% trainable | {r['ms_per_case']:.0f} ms/case")

In [ ]:
SUB_DIR = OUTPUT_DIR / "submissions"
SUB_DIR.mkdir(exist_ok=True)
POOL_TEST = set(POOL_IDS["test"])
print(f"{'run':30s} {'dev Micro-F1@7':>14s}  zip")
for n in ORDER:
    zp = OUTPUT_DIR / "runs" / n / "submission.zip"
    if not zp.exists():
        continue
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
        sub = json.loads(z.read("prediction.json"))
    assert list(sub) == test_ids
    for qid, r in sub.items():
        assert len(r) == TOP_K and len(set(r)) == TOP_K and set(r) <= POOL_TEST and all(isinstance(x, str) for x in r)
    dest = SUB_DIR / f"phase7_{ARCH_NAME}_{n}.zip"
    shutil.copy(zp, dest)
    f1 = RESULTS[n]["dev"]["micro_f1"]
    print(f"{n:30s} {f1:14.4f}  {dest}")
print(f"\nEach zip: prediction.json at the root = {{query_id: [{TOP_K} unique candidate ids from test_candidates.json,")
print("best first]} for every test query. BM25_baseline uses no neural model - submit the Phase-1-based runs; your best")
print("four Codabench submissions must come from different Phase-1 models.")